In [3]:
from pathlib import Path
from datetime import datetime, timedelta
import time
import cdsapi

c = cdsapi.Client()

output_dir = Path(r"D:\Rubel\M.Tech\MTP\Phase 2\era5_downloads")
output_dir.mkdir(parents=True, exist_ok=True)

area = [38, 70, 25, 99]

# CDS daily-statistics requests use whole-hour UTC offsets.
TIME_ZONE = "UTC+00:00"

# Variables that are typically averaged over the day
mean_variables = [
    "2m_temperature",
    "2m_dewpoint_temperature",
    "surface_pressure",
    "snow_depth_water_equivalent",
    "lake_ice_depth",
    "lake_ice_temperature",
    "lake_shape_factor",
]

# Variables that are typically accumulated over the day
sum_variables = [
    "total_precipitation",
    "snowfall",
    "snowmelt",
    "surface_runoff",
    "surface_solar_radiation_downwards",
]

start_year = 1974
end_year = 2025


def latest_complete_month_ist() -> tuple[int, int]:
    # Dataset updates are not real-time; use previous local month as safe bound.
    now_ist = datetime.utcnow() + timedelta(hours=5, minutes=30)
    year = now_ist.year
    month = now_ist.month - 1
    if month == 0:
        year -= 1
        month = 12
    return year, month


def is_beyond_latest_complete_month(year: int, month: int, latest_year: int, latest_month: int) -> bool:
    return (year, month) > (latest_year, latest_month)


def probe_supported_variables(variables: list[str], statistic: str, probe_year: int = 2000, probe_month: int = 1) -> list[str]:
    """Check variable support for this dataset/statistic using tiny one-variable requests."""
    supported = []
    print(f"\nValidating {statistic} variables using {probe_year}-{probe_month:02d}...")

    for var in variables:
        request = {
            "product_type": "reanalysis",
            "variable": [var],
            "year": str(probe_year),
            "month": f"{probe_month:02d}",
            "daily_statistic": f"daily_{statistic}",
            "time_zone": TIME_ZONE,
            "frequency": "1-hourly",
            "area": area,
            "format": "netcdf",
        }

        # Probe target file is overwritten repeatedly to avoid clutter.
        probe_target = output_dir / f"_probe_{statistic}.nc"

        try:
            c.retrieve("derived-era5-land-daily-statistics", request, str(probe_target))
            supported.append(var)
            print(f"  OK   : {var}")
        except Exception as exc:
            print(f"  SKIP : {var}")
            print(f"         Reason: {exc}")

    return supported


def download_month(year: int, month: int, variables: list[str], statistic: str) -> tuple[bool, str | None]:
    output_file = output_dir / f"era5_land_daily_{statistic}_{year}_{month:02d}.nc"

    if output_file.exists():
        print(f"  {year}-{month:02d} [{statistic}] already exists, skipping")
        return True, None

    request = {
        "product_type": "reanalysis",
        "variable": variables,
        "year": str(year),
        "month": f"{month:02d}",
        "daily_statistic": f"daily_{statistic}",
        "time_zone": TIME_ZONE,
        "frequency": "1-hourly",
        "area": area,
        "format": "netcdf",
    }

    last_error = None
    for attempt in range(1, 4):
        try:
            print(f"  Downloading {year}-{month:02d} [{statistic}]...")
            c.retrieve(
                "derived-era5-land-daily-statistics",
                request,
                str(output_file),
            )
            print(f"  Saved: {output_file.name}")
            return True, None
        except Exception as exc:
            last_error = str(exc)
            if attempt == 3:
                print(f"  Failed {year}-{month:02d} [{statistic}]: {exc}")
                return False, last_error

            wait = 10 * attempt
            print(f"  Attempt {attempt} failed, retrying in {wait}s...")
            time.sleep(wait)

    return False, last_error


latest_year, latest_month = latest_complete_month_ist()

print(f"Output folder            : {output_dir}")
print(f"Area                     : {area}")
print(f"Years                    : {start_year}-{end_year}")
print(f"Time zone                : {TIME_ZONE}")
print(f"Latest complete month IST: {latest_year}-{latest_month:02d}")
print(f"Planned requests         : {(end_year - start_year + 1) * 12 * 2}\n")

# Preflight validation prevents repeated failures caused by unsupported variables.
mean_variables = probe_supported_variables(mean_variables, "mean")
sum_variables = probe_supported_variables(sum_variables, "sum")

if not mean_variables:
    raise RuntimeError("No valid mean variables after CDS validation. Check account permissions and dataset variable names.")
if not sum_variables:
    raise RuntimeError("No valid sum variables after CDS validation. Check account permissions and dataset variable names.")

print("\nValidated variable sets:")
print(f"  mean ({len(mean_variables)}): {mean_variables}")
print(f"  sum  ({len(sum_variables)}): {sum_variables}")

failed_requests = []
skipped_unavailable = []
error_reasons = {}

for year in range(start_year, end_year + 1):
    for month in range(1, 13):
        if is_beyond_latest_complete_month(year, month, latest_year, latest_month):
            skipped_unavailable.append(f"{year}-{month:02d}")
            continue

        print(f"\n{year}-{month:02d}")
        ok_mean, err_mean = download_month(year, month, mean_variables, "mean")
        ok_sum, err_sum = download_month(year, month, sum_variables, "sum")

        if not ok_mean:
            key = f"{year}-{month:02d} [mean]"
            failed_requests.append(key)
            if err_mean:
                error_reasons[key] = err_mean
        if not ok_sum:
            key = f"{year}-{month:02d} [sum]"
            failed_requests.append(key)
            if err_sum:
                error_reasons[key] = err_sum

print("\nRun complete.")
print(f"Skipped future/unavailable months: {len(skipped_unavailable)}")
print(f"Failed requests                  : {len(failed_requests)}")
if failed_requests:
    print("First failures:")
    for item in failed_requests[:10]:
        print(f"  - {item}")
        reason = error_reasons.get(item)
        if reason:
            print(f"    Reason: {reason}")

C:\Users\rubel\AppData\Local\Temp\ipykernel_23700\569104204.py:42: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now_ist = datetime.utcnow() + timedelta(hours=5, minutes=30)


Output folder            : D:\Rubel\M.Tech\MTP\Phase 2\era5_downloads
Area                     : [38, 70, 25, 99]
Years                    : 1974-2025
Time zone                : UTC+00:00
Latest complete month IST: 2026-03
Planned requests         : 1248


Validating mean variables using 2000-01...
  SKIP : 2m_temperature
         Reason: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/derived-era5-land-daily-statistics/execution
invalid request
None of the data you have requested is available yet, please revise the period requested. The latest date available for this dataset is: 2026-03-31 10:00
  SKIP : 2m_dewpoint_temperature
         Reason: 400 Client Error: Bad Request for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/derived-era5-land-daily-statistics/execution
invalid request
None of the data you have requested is available yet, please revise the period requested. The latest date available for this dataset is

RuntimeError: No valid mean variables after CDS validation. Check account permissions and dataset variable names.

In [ ]:
import cdsapi
import os

c = cdsapi.Client()

output_dir = "era5_downloads"
os.makedirs(output_dir, exist_ok=True)

for year in range(1974, 2025):
    output_file = f"{output_dir}/era5_land_daily_{year}.nc"

    # Skip if already downloaded — safe to resume if interrupted
    if os.path.exists(output_file):
        print(f"{year} already exists, skipping")
        continue

    print(f"Downloading {year}...")

    try:
        c.retrieve(
            "derived-era5-land-daily-statistics",
            {
                "product_type"   : "reanalysis",
                "variable"       : [
                    "2m_temperature",
                    "total_precipitation",
                    "snowfall",
                    "snow_depth_water_equivalent",
                    "snowmelt",
                    "surface_runoff"
                ],
                "year"           : str(year),
                "month"          : [f"{m:02d}" for m in range(1, 13)],
                "daily_statistic": "daily_mean",
                "time_zone"      : "UTC+00:00",
                "frequency"      : "1-hourly",
                "area"           : [40, 60, 25, 105],
                "format"         : "netcdf"
            },
            output_file
        )
        print(f"  Saved: {output_file}")

    except Exception as e:
        print(f"  Failed for {year}: {e}")
        # Continue to next year — don't crash the whole loop
        continue

2026-04-05 12:20:22,757 INFO Request ID is 8711cf2c-d89c-40c1-bdd6-a241c7aab07b
2026-04-05 12:20:22,956 INFO status has been updated to accepted


In [4]:
import cdsapi

dataset = "derived-era5-single-levels-daily-statistics"
request = {
    "product_type": "reanalysis",
    "variable": [
        "2m_dewpoint_temperature",
        "2m_temperature",
        "surface_pressure",
        "total_precipitation",
        "surface_net_solar_radiation_clear_sky",
        "lake_depth",
        "lake_ice_depth",
        "lake_ice_temperature",
        "lake_shape_factor",
        "runoff",
        "surface_runoff"
    ],
    "year": "2017",
    "month": ["01"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "daily_statistic": "daily_mean",
    "time_zone": "utc+00:00",
    "frequency": "1_hourly",
    "area": [38, 70, 25, 99]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


2026-04-05 17:01:31,334 INFO Request ID is 2c60da08-3b1a-4e41-b7b6-cae1100a18c4
2026-04-05 17:01:31,555 INFO status has been updated to accepted
2026-04-05 17:03:27,546 INFO status has been updated to running
2026-04-05 17:20:03,758 INFO status has been updated to successful


b0795eb09d74b2b197bb28578302e398.zip:   0%|          | 0.00/3.57M [00:00<?, ?B/s]

'b0795eb09d74b2b197bb28578302e398.zip'

In [5]:
import cdsapi

dataset = "derived-era5-single-levels-daily-statistics"
request = {
    "product_type": "reanalysis",
    "variable": [
        # Keep from your original
        "2m_dewpoint_temperature",
        "2m_temperature",
        "surface_pressure",
        "total_precipitation",
        "surface_runoff",
        "surface_net_solar_radiation_clear_sky",
        # Add missing critical variables
        "snowfall",
        "snowmelt",
        "snow_depth",
        "sub_surface_runoff",
    ],
    "year": "2017",
    "month": ["01"],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "daily_statistic": "daily_mean",   # change to daily_sum for precip/snowfall/runoff
    "time_zone": "UTC+00:00",
    "frequency": "1_hourly",
    "area": [38, 70, 25, 99]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()

2026-04-05 17:40:56,240 INFO Request ID is c57688d3-5a15-445e-bf13-958016397bd9
2026-04-05 17:40:56,438 INFO status has been updated to accepted
Recovering from connection error [HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/jobs/c57688d3-5a15-445e-bf13-958016397bd9?log=True&request=True (Caused by NameResolutionError("HTTPSConnection(host='cds.climate.copernicus.eu', port=443): Failed to resolve 'cds.climate.copernicus.eu' ([Errno 11001] getaddrinfo failed)"))], attempt 1 of 500
Retrying in 120 seconds


SSLError: HTTPSConnectionPool(host='cds.climate.copernicus.eu', port=443): Max retries exceeded with url: /api/retrieve/v1/jobs/c57688d3-5a15-445e-bf13-958016397bd9?log=True&request=True (Caused by SSLError(SSLError(1, '[SSL] record layer failure (_ssl.c:1010)')))

In [11]:
import cdsapi
import time
from pathlib import Path
from itertools import product

client = cdsapi.Client()

dataset = "derived-era5-single-levels-daily-statistics"
AREA = [38, 70, 25, 99]
OUTPUT_DIR = Path("era5_daily")
OUTPUT_DIR.mkdir(exist_ok=True)

VARIABLE_GROUPS = {
    "daily_mean": [
        "2m_temperature",
        "2m_dewpoint_temperature",
        "surface_pressure",
        "snow_depth",
    ],
    "daily_sum": [
        "total_precipitation",
        "snowfall",
        "snowmelt",
        "surface_runoff",
        "sub_surface_runoff",
    ],
}

# Quarters: (label, months)
QUARTERS = {
    "Q1": ["01", "02", "03"],
    "Q2": ["04", "05", "06"],
    "Q3": ["07", "08", "09"],
    "Q4": ["10", "11", "12"],
}

ALL_DAYS = [f"{d:02d}" for d in range(1, 32)]
YEARS = [str(y) for y in range(1974, 1976)]


def output_path(year, quarter, statistic):
    return OUTPUT_DIR / f"era5_{statistic}_{year}_{quarter}.nc"


def build_request(year, months, statistic, variables):
    return {
        "product_type": "reanalysis",
        "variable": variables,
        "year": year,
        "month": months,
        "day": ALL_DAYS,
        "daily_statistic": statistic,
        "time_zone": "UTC+00:00",
        "frequency": "1_hourly",
        "area": AREA,
    }


# --- Submit up to MAX_ACTIVE jobs at a time, poll the rest ---
MAX_ACTIVE = 2  # CDS free tier limit
pending = []    # (out_path, result_handle)

def submit_job(year, quarter, months, statistic, variables):
    out = output_path(year, quarter, statistic)
    if out.exists():
        print(f"  SKIP (exists): {out.name}")
        return None
    print(f"  Submitting: {year} {quarter} | {statistic}")
    result = client.retrieve(dataset, build_request(year, months, statistic, variables))
    return (out, result)


# Build full job list
all_jobs = list(product(YEARS, QUARTERS.items(), VARIABLE_GROUPS.items()))
# all_jobs entries: (year, (q_label, months), (statistic, variables))

print(f"Total jobs to process: {len(all_jobs)}\n")

submitted = []

for year, (q_label, months), (statistic, variables) in all_jobs:
    job = submit_job(year, q_label, months, statistic, variables)
    if job:
        submitted.append(job)

    # Throttle: don't flood the queue
    time.sleep(3)

# --- Download all in order (they finish asynchronously on CDS side) ---
print(f"\n{len(submitted)} jobs submitted. Downloading as they complete...\n")

for out, result in submitted:
    try:
        print(f"Waiting for: {out.name}")
        result.download(str(out))
        print(f"  ✓ Done: {out.name} ({out.stat().st_size / 1e6:.1f} MB)")
    except Exception as e:
        print(f"  ✗ FAILED: {out.name} — {e}")
        # Log failed jobs for retry
        with open(OUTPUT_DIR / "failed_jobs.txt", "a") as f:
            f.write(f"{out.name}\n")

Total jobs to process: 16

  Submitting: 1974 Q1 | daily_mean


2026-04-06 00:46:28,478 INFO Request ID is 5f281658-1950-4042-bc53-c1e1a964634c
2026-04-06 00:46:28,681 INFO status has been updated to accepted
2026-04-06 10:38:25,137 INFO status has been updated to successful


  Submitting: 1974 Q1 | daily_sum


HTTPError: 403 Client Error: Forbidden for url: https://cds.climate.copernicus.eu/api/retrieve/v1/processes/derived-era5-single-levels-daily-statistics/execution
cost limits exceeded
Your request is too large, please reduce your selection.

In [13]:
from osgeo import gdal


In [14]:
ds = gdal.Open(r"C:\Users\rubel\Downloads\era5_land_hma_1981.tif")
ds

d:\anaconda\geo_dl\Lib\site-packages\osgeo\gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


<osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x0000028BA6D81170> >

In [ ]:
import xarray as xr

# Load the file
# 'mask_and_scale=True' handles the nodata values
ds = xr.open_dataset(r"C:\Users\rubel\Downloads\era5_land_hma_1981.tif", engine="rasterio")

# Your bands are likely named 'band_1', 'band_2', etc.
# You can select a specific band (e.g., the first one) and plot it
ds.band_1.plot()

<bound method MajorObject.GetDescription of <osgeo.gdal.Dataset; proxy of <Swig Object of type 'GDALDatasetShadow *' at 0x0000028BA6D81170> >>


In [24]:
b1 = ds.GetRasterBand(1).ReadAsArray()
b1

array([[268.1119353 , 269.89025561, 270.26476733, ..., 257.53210131,
        256.88879077, 256.08402189],
       [270.90726407, 270.72301928, 269.55675952, ..., 256.80635262,
        256.94632657, 256.08410327],
       [269.77095222, 267.69616381, 266.56961759, ..., 257.27038256,
        257.97326342, 258.37267749],
       ...,
       [292.67533048, 292.87650235, 292.96870613, ..., 284.07238452,
        286.44258308, 283.75825691],
       [292.85200691, 292.97961108, 293.04308764, ..., 284.64367358,
        286.86795743, 284.67231941],
       [293.04219246, 293.09810066, 293.10192553, ..., 285.45169767,
        287.34842618, 285.29170418]], shape=(141, 291))